# llm-finetune-serve — Colab driver

This notebook stays thin on purpose: clone, install, call scripts. All logic
lives in `src/`. If you find yourself writing real code here, it belongs in the
repo instead.

## Two ways to run it

**In the browser:** open this notebook from GitHub, then Runtime → Change
runtime type → GPU.

**From VS Code** (no browser tab): install the official **Google Colab**
extension (publisher: Google), open this file locally, then kernel picker →
`Colab` → `Auto Connect`, and pick a GPU runtime.

Either way the kernel runs on a Colab VM, and the extension does **not** sync
local files to it — so your `src/` edits reach the GPU through GitHub. The loop
is: edit locally → commit + push → re-run the `git pull` cell below → re-run
the stage cell.

In [ ]:
!nvidia-smi

## 1. Clone the repo

Public repo, so no credentials are needed. Re-running this cell pulls the
latest commit rather than re-cloning.

(If you ever flip it back to private, add a GitHub PAT with `repo` scope as a
Colab secret named `GITHUB_TOKEN` — the cell picks it up automatically.)

In [ ]:
OWNER = "rushilpatra"
REPO = "llm-finetune-serve"
BRANCH = "main"

import os, subprocess

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = os.environ.get("GITHUB_TOKEN")

auth = f"{TOKEN}@" if TOKEN else ""
url = f"https://{auth}github.com/{OWNER}/{REPO}.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO], check=True)
%cd /content/llm-finetune-serve
!git pull --ff-only

## 2. Install

vLLM resolves its own torch build, so it goes first and the rest follows.
Colab will warn about a session restart — restart, then re-run the `%cd` cell
above and continue from here.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    # T4 (Turing) has no bf16 support; scripts detect this at runtime.
    print("gpu         ", name)
    print("bf16        ", torch.cuda.is_bf16_supported())

## 3. Data smoke test

Prints split sizes, the 8-shot prefix length, and a sample prompt with the
round-trip answer-extraction check.

In [ ]:
!python -m src.data --split val --limit 2

## 4. Baseline: 8-shot prompting of the base model

Plumbing check first (no GPU, seconds), then the real run. Predictions stream
to `results/baseline_8shot.jsonl` as they are produced, so if the session dies
you re-run the same command and it picks up where it stopped.

In [ ]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --dry-run --limit 4

In [ ]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml

## 5. Save results off the VM

Colab VMs are ephemeral. Download `results/` and commit it from your laptop —
the per-example JSONL is what the paired bootstrap reads later.

In [ ]:
!cd /content/llm-finetune-serve && zip -qr /content/results.zip results

from google.colab import files
files.download("/content/results.zip")

## Next stages

Cells for training, merging, and benchmarking get added here as those scripts
land. Each is a single `!python -m src.<script> --config configs/<run>.yaml`.